%% [markdown]<br>
# Sequence Processing with HMMs and CRFs<br>
<br>
**The goal of this practical is to study sequence models in NLP.**<br>
<br>
We will work on Part-Of-Speech (POS) and optionally on chunking (gathering different groups in sentences). The datasets are from [CONLL 2000](https://www.clips.uantwerpen.be/conll2000/chunking/): <br>
- **Small corpus:** chtrain/chtest to understand the tools and models <br>
- **Larger corpus:** train/test to collect reliable experimental results<br>
<br>
<br>
# 1) HMMS<br>


%%

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import seaborn as sns
from sklearn.metrics import confusion_matrix

%%<br>
Loading POS/Chunking data

In [ ]:
def load(filename):
    listeDoc = list()
    with open(filename, "r") as f:
        doc = list()
        for ligne in f:
            if len(ligne) < 2: # fin de doc
                listeDoc.append(doc)
                doc = list()
                continue
            mots = ligne.replace("\n","").split(" ")
            doc.append((mots[0],mots[1])) # Change mots[1] -> mots[2] for chuncking
    return listeDoc

%%<br>
=============== loding ============<br>
small corpus => ideal for first tests

In [ ]:
filename = "./datasets/conll2000/chtrain.txt" 
filenameT = "./datasets/conll2000/chtest.txt" 

In [ ]:
alldocs = load(filename)
alldocsT = load(filenameT)

In [ ]:
print(len(alldocs)," docs read")
print(len(alldocsT)," docs (T) read")

%%

In [ ]:
print(alldocs[0])
print(alldocsT[0])

%% [markdown]<br>
## Building a baseline POS model (without sequence)<br>
<br>
We will build a simple dictionary ```word => PoS label``` without taking into account any sequence information. We will compare the sequence models to this baseline.

%%<br>
Dictionary building 

In [ ]:
dico = dict()
word_pos_counts = defaultdict(lambda: defaultdict(int))

Count the occurrences of each POS tag for each word

In [ ]:
for doc in alldocs:
    for word, pos in doc:
        word_pos_counts[word][pos] += 1

Assign the most frequent POS tag to the word

In [ ]:
for word, pos_counts in word_pos_counts.items():
    best_pos = max(pos_counts, key=pos_counts.get)
    dico[word] = best_pos

%%<br>
Evaluate test performances

In [ ]:
correct = 0
total = 0
default_pos = 'NN' # Majority class

In [ ]:
for doc in alldocsT:
    for word, true_pos in doc:
        # Predict using dictionary, fallback to default_pos if unknown
        pred_pos = dico.get(word, default_pos)
        if pred_pos == true_pos:
            correct += 1
        total += 1

In [ ]:
print(f"Baseline accuracy: {correct}/{total} ({correct/total*100:.2f}%)")

%%[markdown]<br>
## HMMs

%%

In [ ]:
def learnHMM(allx, allq, N, K, initTo1=True):
    if initTo1:
        eps = 1e-1 
        A = np.ones((N,N))*eps
        B = np.ones((N,K))*eps
        Pi = np.ones(N)*eps
    else:
        A = np.zeros((N,N))
        B = np.zeros((N,K))
        Pi = np.zeros(N)
    for x,q in zip(allx,allq):
        if len(q) == 0: continue
        Pi[int(q[0])] += 1
        for i in range(len(q)-1):
            A[int(q[i]),int(q[i+1])] += 1
            B[int(q[i]),int(x[i])] += 1
        B[int(q[-1]),int(x[-1])] += 1 # last transition
    A = A/np.maximum(A.sum(1).reshape(N,1),1) # normalisation
    B = B/np.maximum(B.sum(1).reshape(N,1),1) # normalisation
    Pi = Pi/Pi.sum()
    return Pi , A, B

In [ ]:
def viterbi(x,Pi,A,B):
    T = len(x)
    N = len(Pi)
    if T == 0: return[], 0
    logA = np.log(A)
    logB = np.log(B)
    logdelta = np.zeros((N,T))
    psi = np.zeros((N,T), dtype=int)
    S = np.zeros(T)
    logdelta[:,0] = np.log(Pi) + logB[:,int(x[0])]
    #forward
    for t in range(1,T):
        logdelta[:,t] = (logdelta[:,t-1].reshape(N,1) + logA).max(0) + logB[:,int(x[t])]
        psi[:,t] = (logdelta[:,t-1].reshape(N,1) + logA).argmax(0)
    # backward
    logp = logdelta[:,-1].max()
    S[T-1] = logdelta[:,-1].argmax()
    for i in range(2,T+1):
        S[int(T-i)] = psi[int(S[int(T-i+1)]),int(T-i+1)]
    return S, logp

%% [markdown]<br>
### Data encoding

%%

In [ ]:
buf = [[m for m,pos in d ] for d in alldocs]
mots = [][mots.extend(b) for b in buf]
mots = np.unique(np.array(mots))
nMots = len(mots)+1 # mot inconnu

In [ ]:
mots2ind = dict(zip(mots,range(len(mots))))
mots2ind["UUUUUUUU"] = len(mots)

In [ ]:
buf2 = [[pos for m,pos in d ] for d in alldocs]
cles =[]
[cles.extend(b) for b in buf2]
cles = np.unique(np.array(cles))
cles2ind = dict(zip(cles,range(len(cles))))

In [ ]:
nCles = len(cles)

In [ ]:
print(nMots,nCles," in the dictionary")

mise en forme des donnÃ©es

In [ ]:
allx  = [[mots2ind[m] for m,pos in d] for d in alldocs]
allxT = [[mots2ind.setdefault(m,len(mots)) for m,pos in d] for d in alldocsT]

In [ ]:
allq  = [[cles2ind[pos] for m,pos in d] for d in alldocs]
allqT = [[cles2ind.setdefault(pos,len(cles)) for m,pos in d] for d in alldocsT]

%%<br>
First doc:

In [ ]:
print(allx[0])
print(allq[0])

%% [markdown]<br>
## You turn: apply HMMs to those data!

%%<br>
HMM training 

In [ ]:
Pi, A, B = learnHMM(allx, allq, nCles, nMots, initTo1=True)

%%<br>
HMM decoding and performances evaluation

In [ ]:
correct_hmm = 0
total_hmm = 0
predictions =[]

In [ ]:
for x, q in zip(allxT, allqT):
    if len(x) == 0: 
        continue
    pred_seq, _ = viterbi(x, Pi, A, B)
    predictions.append(pred_seq)
    
    for p, true_q in zip(pred_seq, q):
        if int(p) == int(true_q):
            correct_hmm += 1
        total_hmm += 1

In [ ]:
print(f"HMM accuracy: {correct_hmm}/{total_hmm} ({correct_hmm/total_hmm*100:.2f}%)")

%%[markdown]<br>
### Qualitative Analyis:

%%<br>
1. Visualization of Transition Matrix A

In [ ]:
plt.figure(figsize=(12, 10))
plt.imshow(A, cmap='viridis', interpolation='nearest')
plt.title("HMM Transition Matrix (A)")
plt.colorbar()
# Show a few labels for clarity
plt.xticks(np.arange(nCles), cles, rotation=90, fontsize=8)
plt.yticks(np.arange(nCles), cles, fontsize=8)
plt.show()

2. Confusion Matrix

In [ ]:
flat_true = [cles[int(q)] for seq in allqT for q in seq]
flat_pred = [cles[int(p)] for seq in predictions for p in seq]

In [ ]:
cm = confusion_matrix(flat_true, flat_pred, labels=cles)
plt.figure(figsize=(15, 12))
sns.heatmap(cm, xticklabels=cles, yticklabels=cles, cmap='Blues', annot=False)
plt.title("Confusion Matrix for HMM Predictions")
plt.xlabel("Predicted POS")
plt.ylabel("True POS")
plt.show()

3. Find out examples that are corrected by Viterbi decoding

In [ ]:
print("Examples corrected by Viterbi (Baseline failed, HMM succeeded):")
found = 0
for doc, seq_x, seq_q, seq_pred in zip(alldocsT, allxT, allqT, predictions):
    for i in range(len(doc)):
        word, true_pos = doc[i]
        baseline_pred = dico.get(word, 'NN')
        viterbi_pred = cles[int(seq_pred[i])]
        
        if baseline_pred != true_pos and viterbi_pred == true_pos:
            print(f"Word: '{word:12}' | True: {true_pos:4} | Baseline: {baseline_pred:4} | Viterbi: {viterbi_pred:4}")
            found += 1
            if found >= 10: break
    if found >= 10: break

%% [markdown]<br>
# 2) Conditional Random Fields (CRF)

%%<br>
!pip install python-crfsuite

In [ ]:
from nltk.tag.crf import CRFTagger
import os
os.makedirs('out', exist_ok=True)

In [ ]:
tagger = CRFTagger()
tagger.train(alldocs, 'out/crf.model') # training

Evaluation of CRF

In [ ]:
crf_preds = tagger.tag_sents([[m for m, pos in doc] for doc in alldocsT])
correct_crf = 0
total_crf = 0

In [ ]:
for doc_true, doc_pred in zip(alldocsT, crf_preds):
    for (_, true_pos), (_, pred_pos) in zip(doc_true, doc_pred):
        if true_pos == pred_pos:
            correct_crf += 1
        total_crf += 1

In [ ]:
print(f"CRF accuracy: {correct_crf}/{total_crf} ({correct_crf/total_crf*100:.2f}%)")

%%<br>
perceptron

In [ ]:
from nltk.tag.perceptron import PerceptronTagger
tagger_perc = PerceptronTagger(load=False)
tagger_perc.train(alldocs)

Evaluation of Perceptron

In [ ]:
perc_preds = tagger_perc.tag_sents([[m for m, pos in doc] for doc in alldocsT])
correct_perc = 0
total_perc = 0

In [ ]:
for doc_true, doc_pred in zip(alldocsT, perc_preds):
    for (_, true_pos), (_, pred_pos) in zip(doc_true, doc_pred):
        if true_pos == pred_pos:
            correct_perc += 1
        total_perc += 1

In [ ]:
print(f"Perceptron accuracy: {correct_perc}/{total_perc} ({correct_perc/total_perc*100:.2f}%)")